In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


In this notebook I will load the neural data from the EMU cohort, and make figures for the paper:
    

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import mne
import mne_connectivity
from mne_connectivity import phase_slope_index, seed_target_indices, spectral_connectivity_epochs
from glob import glob
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import zscore, linregress, ttest_ind, ttest_rel, ttest_1samp
import pandas as pd
from mne.preprocessing.bads import _find_outliers
import fooof
# Import plotting functions
from fooof.plts.spectra import plot_spectrum, plot_spectra
# Import the FOOOF object
from fooof import FOOOF
from fooof import FOOOFGroup
import os 
import joblib
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from tqdm import tqdm
from IPython.display import clear_output
import warnings 

# I only want to see warnings once
warnings.filterwarnings('ignore')

from joblib import delayed, Parallel
from statsmodels.stats import multitest

from neurodsp.plts import plot_power_spectra

In [ ]:
import sys
sys.path.append('/hpc/users/qasims01/resources/LFPAnalysis')

In [ ]:
from LFPAnalysis import lfp_preprocess_utils, sync_utils, analysis_utils, nlx_utils, oscillation_utils, statistics_utils


In [ ]:
# Set some plot defaults for matplotlib (when not using seaborn): 

import  matplotlib.font_manager

path = '/sc/arion/projects/guLab/Salman/Arial.ttf'
matplotlib.font_manager.fontManager.addfont(path)

prop = matplotlib.font_manager.FontProperties(fname=path)

# Set the default font family to the registered custom font
plt.rcParams['font.family'] = prop.get_name()

# Rest of your customization code
plt.rcParams['font.size'] = 20
plt.rcParams['lines.linewidth'] = 1

# Set palette: 

# Define RGB values for your two colors
ignacio_purple = np.array([86, 28, 70]) / 255
ignacio_blue = np.array([0, 70, 126]) / 255 

# Create a Seaborn palette with your custom colors normalized to 0-1
custom_palette = sns.color_palette([ignacio_purple, ignacio_blue])

ignacio_orange = np.array([255, 172, 77]) / 255
ignacio_mauve = np.array([193, 155, 211]) / 255
ignacio_beige = np.array([205, 182, 162]) / 255

# Set the custom palette
sns.set_palette(custom_palette)
sns.set_context("paper")

# from YBA
hpc_c = np.array([0.44314, 0.2902, 1])
amy_c = np.array([0.49412, 0, 0.90196])
ofc_c = np.array([1, 0.75686, 0.41569])
acc_c = np.array([0.7098, 0.27059 ,0.68235])
dmpfc_c = np.array([0.40784, 0.01961, 0])
dlpfc_c = np.array([1, 0.06275, 0.06275])


In [ ]:
# plot the brain region swatches for figure design

In [ ]:
# Create a color palette
palette = sns.color_palette([dmpfc_c, dlpfc_c, ofc_c, acc_c, amy_c, hpc_c])

# Display the color swatches
sns.palplot(palette)
plt.show()

In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site!='UI']

# Let's get rid of the patients who do worse than chance on the DM task, or have no hits in the memory task

subj_df = subj_df[~subj_df['MSSMCode'].isin(['UI002', 'MS024', 'MS034', 'MS038'])]

days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df.csv')
combined_df = pd.read_csv(f'{load_dir}/combined_df.csv')

# Filter just to the participants in this notebook
learn_df = learn_df[learn_df.participant.isin(subj_ids)]
combined_df = combined_df[combined_df.participant.isin(subj_ids)]


# I enter the day information in stupidly 
combined_df.condition = combined_df.condition.apply(lambda x: 'Day1' if x=='day_1' else 'Day2' if x=='day_2' else np.nan)
combined_df['recog_time'] = combined_df['recog_time'].apply(lambda x: float(x) if (~pd.isna(x)) & (x!='None') else x)
combined_df['baseline_start_mem'] = combined_df['baseline_start_mem'].apply(lambda x: float(x) if (~pd.isna(x)) & (x!='None') else x)

# # add RPE to the combined df dataframe: 

combined_df['rpe'] = np.nan
for ix, row in learn_df.iterrows():
    combined_df.loc[(combined_df.img_path==row.img_path) & (combined_df.participant==row.participant), 'rpe'] = row.rpe


## Figure 3: Hippocampal oscillations encode RPE, while hippocampal HFA tracks memory encoding 

Let's: 

1. load the TFRs

2. restrict them to alpha, beta, and HFA

3. average within band

4. split by RPE magnitude 

5. plot

In [ ]:
# Explicitly define a list of analysis events and the baseline event. Should correspond to the dict


evs = {'gamble_start':[-1.0, 0], 
       'feedback_start': [0, 1.5],
       'baseline_start': [0, 0.75]}

baseline_ev = 'baseline_start'
analysis_evs = ['feedback_start']

day = 'Day1'

for i in analysis_evs: 
    if i not in evs.keys(): 
        raise ValueError('This analysis event is not in the dictionary')
        
# Define parameters for TFR: 
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

In [ ]:
frequency_masks = {'theta': freqs<10,
                   'alpha': (freqs>=10) & (freqs<14),
                   'beta': (freqs>=14) & (freqs<30), 
                   'hfa': (freqs>=65)}
                   

In [ ]:
rois = ['HPC', 'OFC', 'dmPFC', 'ACC', 'AMY']

For plotting the power in time (not very effective): 

In [ ]:
# # Let's load the power epochs: 

# analysis_evs = ['feedback_start']

# # task condition to contrast 
# conditions = ['(rpe>0)',
#              '(rpe<0)']

# cond_name = 'rpe'

# tfr_group_data = []
# # {f'{a}': {f'{b}': [] for b in conditions} for a in rois}

# power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids}

# for subj_id in subj_ids: 
#     for event in analysis_evs:
#         filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
#         power_epochs[subj_id][event] = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
                
#         power_epochs[subj_id][event].metadata = learn_df[(learn_df.participant==subj_id)]
        
#         times = power_epochs[subj_id][event].times
            
#         # Get electrode df 
#         elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
        
#         progress_bar = tqdm(elec_df.label, ascii=True, desc='Aggregating TFRs')

#         for elec_ix, chan in enumerate(progress_bar):
#             region = elec_df.salman_region.iloc[elec_ix]
#             if region in rois:
#                 for ix, cond in enumerate(conditions):
#                     for band in frequency_masks.keys():
#                         # Set the times for 
#                         times = power_epochs[subj_id][event].times
#                         plot_data = np.nanmean(np.nanmean(power_epochs[subj_id][event][cond].pick_channels([chan]).data, axis=0), axis=0)
#                         plot_data = np.nanmean(plot_data[frequency_masks[band], :], axis=0)
#                         plot_df = pd.DataFrame(columns=['times', 'power', 'band', 'condition', 'region'])
#                         plot_df['times'] = times
#                         plot_df['power'] = plot_data
#                         plot_df['band'] = band
#                         plot_df['condition'] = cond
#                         plot_df['region'] = region 
#                         plot_df['participant'] = subj_id
#                         tfr_group_data.append(plot_df)                    
                    

In [ ]:
# plot_dfs = pd.concat(tfr_group_data)

In [ ]:
# sns.lineplot(data=plot_dfs[plot_dfs.band=='beta'], x='times', y='power', hue='condition', errorbar='se')


In [ ]:
progress_bar = tqdm(subj_ids, ascii=True, desc='Saving TFR dataframes')
analysis_evs = ['feedback_start'] 

power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids}


for subj_id in subj_ids:
    for day in ['Day1']:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
        for event in analysis_evs:

            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            if not os.path.exists(f'{filepath}/{event}/plots'):
                os.makedirs(f'{filepath}/{event}/plots')
            if not os.path.exists(f'{filepath}/{event}/dfs'):
                os.makedirs(f'{filepath}/{event}/dfs')

            power_epochs[subj_id][event] = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]

            # determine which epochs to analyze (all of them, in my case)
            # These epochs only grab all images from the gambling task: 

#             epochs_to_analyze = learn_df[(learn_df.participant==subj_id)].trials.values - 1

#             power_epochs[subj_id][event] = power_epochs[subj_id][event][epochs_to_analyze.astype(int)]

#             power_epochs[subj_id][event].metadata = learn_df[(learn_df.participant==subj_id)]

#             # These epochs only grab those images that were later involved in memory recognition: 
            epochs_to_analyze = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_gamble').reset_index(drop=True).trials_gamble.values - 1

            power_epochs[subj_id][event] = power_epochs[subj_id][event][epochs_to_analyze.astype(int)]

            power_epochs[subj_id][event].metadata = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_gamble').reset_index(drop=True)

            for chan in power_epochs[subj_id][event].ch_names:
                pow_df = power_epochs[subj_id][event].copy().pick_channels([chan]).to_data_frame()
                pow_df['fband'] = pow_df.freq.apply(lambda x: 'theta' if x<10 else 'alpha' if (x>=10) & (x<14) else 'beta' if (x>=14) & (x<30) else 'slowgamma' if (x>=30) & (x<55) else 'hfa')
                pow_df = pow_df.groupby(['fband','epoch']).mean().reset_index()
                
                tfr_df = pow_df.drop(columns=['time', 'freq']).rename(columns={'epoch':'trial', f'{chan}':'tfr'})

                power_epochs[subj_id][event].metadata['trial'] = tfr_df['trial'].unique()
                tfr_df = tfr_df.merge(power_epochs[subj_id][event].metadata, on=['trial'])

                tfr_df.insert(0,'channel', chan)
                tfr_df.insert(0,'region', elec_df[elec_df.label==chan].salman_region.values[0])
#                 tfr_df.insert(0,'participant', subj_id)
                tfr_df['event'] = event

                tfr_df.to_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_Day1.csv', index=False)


In [ ]:
chan_dfs = [] 
glms = []
elec_counter = 0 
for subj_id in tqdm(subj_ids):
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
    for event in analysis_evs:
        for chan in elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
            chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')
                
            
            chan_df['unique_elec_id'] = elec_counter
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
            try:
                chan_df['gamble_rt_log'] = np.log(chan_df['gamble_rt'])
            except KeyError:
                chan_df['gamble_rt_log'] = np.log(chan_df['gamble_rt_gamble'])
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            elec_counter +=1
            
            chan_dfs.append(chan_df)
        
chan_dfs = pd.concat(chan_dfs)

In [ ]:
plot_df = chan_dfs[chan_dfs.fband=='hfa']

First, let's plot the relationship between RPE and hippocampal power 

In [ ]:
# sns.lineplot(data=, x='times', y='power', hue='condition', errorbar='se')
f, ax = plt.subplots(1, 4, dpi=300, figsize=(12,3), sharex=True, sharey=True)


sns.regplot(data=plot_df[(plot_df.region=='HPC') & (plot_df.hits==True)].reset_index(drop=True),
            color = custom_palette[0], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[0])
sns.regplot(data=plot_df[(plot_df.region=='HPC') & (plot_df.hits==False)].reset_index(drop=True),
            color = custom_palette[1], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[0])

sns.regplot(data=plot_df[(plot_df.region=='dmPFC') & (plot_df.hits==True)].reset_index(drop=True),
            color = custom_palette[0], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[1])
sns.regplot(data=plot_df[(plot_df.region=='dmPFC') & (plot_df.hits==False)].reset_index(drop=True),
            color = custom_palette[1], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[1])

sns.regplot(data=plot_df[(plot_df.region=='OFC') & (plot_df.hits==True)].reset_index(drop=True), 
            color = custom_palette[0], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[2])
sns.regplot(data=plot_df[(plot_df.region=='OFC') & (plot_df.hits==False)].reset_index(drop=True), 
            color = custom_palette[1], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[2])


sns.regplot(data=plot_df[(plot_df.region=='AMY') & (plot_df.hits==True)].reset_index(drop=True), 
            color = custom_palette[0], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[3])
sns.regplot(data=plot_df[(plot_df.region=='AMY') & (plot_df.hits==False)].reset_index(drop=True), 
            color = custom_palette[1], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[3])

sns.despine()
f.tight_layout()

In [ ]:
# sns.lineplot(data=, x='times', y='power', hue='condition', errorbar='se')
f, ax = plt.subplots(1, 4, dpi=300, figsize=(12,3), sharex=True, sharey=True)


sns.regplot(data=plot_df[(plot_df.region=='HPC') & (plot_df.hits==True)].reset_index(drop=True),
            color = custom_palette[0], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[0])
sns.regplot(data=plot_df[(plot_df.region=='HPC') & (plot_df.hits==False)].reset_index(drop=True),
            color = custom_palette[1], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[0])

sns.regplot(data=plot_df[(plot_df.region=='dmPFC') & (plot_df.hits==True)].reset_index(drop=True),
            color = custom_palette[0], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[1])
sns.regplot(data=plot_df[(plot_df.region=='dmPFC') & (plot_df.hits==False)].reset_index(drop=True),
            color = custom_palette[1], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[1])

sns.regplot(data=plot_df[(plot_df.region=='OFC') & (plot_df.hits==True)].reset_index(drop=True), 
            color = custom_palette[0], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[2])
sns.regplot(data=plot_df[(plot_df.region=='OFC') & (plot_df.hits==False)].reset_index(drop=True), 
            color = custom_palette[1], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[2])


sns.regplot(data=plot_df[(plot_df.region=='AMY') & (plot_df.hits==True)].reset_index(drop=True), 
            color = custom_palette[0], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[3])
sns.regplot(data=plot_df[(plot_df.region=='AMY') & (plot_df.hits==False)].reset_index(drop=True), 
            color = custom_palette[1], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[3])

sns.despine()
f.tight_layout()

Plot the PSDs for every region 

In [ ]:
rois = ['OFC', 'dmPFC', 'HPC', 'AMY', 'ACC']

In [ ]:
# Let's load the power epochs for all : 

evs = {'gamble_start':[-1.0, 0], 
       'feedback_start': [0, 1.5],
       'baseline_start': [0, 0.75]}

baseline_ev = 'baseline_start'
analysis_evs = ['feedback_start']

day = 'Day1'

# Define parameters for TFR: 
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

tfr_group_data = {f'{a}': [] for a in rois}

power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids}

for subj_id in subj_ids: 
    for event in analysis_evs:
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        
        save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/Day1'
        event_epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif', preload=True)
        ch_names = event_epochs_reref.ch_names
        #epochs_all_subjs_all_evs[subj_id][event]
            
        # Let's make sure we only do this for good channels
        good_chans = [x for x in event_epochs_reref.ch_names if x not in event_epochs_reref.info['bads']]
        picks = [x for x in good_chans]
        
        pow_struct = np.nan * np.ones([event_epochs_reref._data.shape[0], 
                               event_epochs_reref._data.shape[1], len(freqs), 
                               event_epochs_reref._data.shape[-1]])
    
        progress_bar = tqdm(np.arange(event_epochs_reref._data.shape[1]), ascii=True, desc='Computing TFR')

        for ch_ix in progress_bar: 
            ch_data = event_epochs_reref._data[:, ch_ix:ch_ix+1, :]
            bad_epochs  = np.where(event_epochs_reref.metadata[event_epochs_reref.ch_names[ch_ix]].notnull())[0]
            good_epochs = np.delete(np.arange(ch_data.shape[0]), bad_epochs)
            ch_data = np.delete(ch_data, bad_epochs, axis=0)
            ch_pow = mne.time_frequency.tfr_array_morlet(ch_data, sfreq=event_epochs_reref.info['sfreq'], 
                                                freqs=freqs, n_cycles=n_cycles, zero_mean=False, 
                                                use_fft=True, output='power', n_jobs=-1)

            pow_struct[good_epochs, ch_ix, :, :] = ch_pow[:, 0, :, :]
        
        temp_pow = mne.time_frequency.EpochsTFR(event_epochs_reref.info, pow_struct, 
                                                event_epochs_reref.times, freqs)
        
        temp_pow.crop(tmin=evs[event][0], tmax=evs[event][1])
        
        # Get electrode df 
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
        
        progress_bar = tqdm(elec_df.label, ascii=True, desc='Aggregating TFRs')

        for elec_ix, chan in enumerate(progress_bar):
            region = elec_df.salman_region.iloc[elec_ix]
            if region in rois:
                # average across channels, events, and times
                plot_data = np.nanmean(np.nanmean(np.nanmean(temp_pow.copy().pick([chan]).data, axis=0), axis=0), axis=-1)
                tfr_group_data[region].append(plot_data)                    
                    

In [ ]:
def normalize_list(input_list):
    min_val = np.nanmin(input_list)
    max_val = np.nanmax(input_list)
    den = (max_val - min_val)
    if den == 0: 
        den = 0.001
    
    normalized_list = [(x - min_val) / den  for x in input_list]
    
    return normalized_list

In [ ]:
freqs = np.logspace(*np.log10([2, 200]), num=30)
save_dir_plots = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBanditsEMU_Ephys'
os.makedirs(save_dir_plots, exist_ok=True)

for region in rois: 

    fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
    # Create a Pandas DataFrame
    plot_data = np.vstack(tfr_group_data[region])
    normalized_array = np.array([normalize_list(lst) for lst in plot_data])
    for ch in normalized_array:
        # plt.plot(freqs[:-1], np.log10(ch)[:-1], alpha=0.1, color='k')
        plot_power_spectra(freqs[:-1], ch[:-1], alpha =0.1,  
                       ax=ax, colors='black')   
    plot_power_spectra(freqs[:-1], np.nanmean(normalized_array[:, :-1], axis=0), 
                       ax=ax, colors='black')   
    # plt.plot(freqs[:-1], np.log10(np.nanmean(normalized_array[:, :-1], axis=0)), alpha=1, color='k')

    
    ax.set(title=f'{region}', xlabel='log frequency (hz)', 
           ylabel='log power (norm)', 
           xticks=[2, 10, 30, 50, 200], 
           xticklabels=[2, 10, 30, 50, 200])
    
    sns.despine()
    
    plt.savefig(f'{save_dir_plots}/feedback_PSD_{region}.pdf', dpi=300)

        
        

In [ ]:
# Let's load the power epochs for all : 

evs = {'recog_time':[-1.0, 0]}

# baseline_ev = 'baseline_start'
analysis_evs = ['recog_time']

day = 'Day1'

# Define parameters for TFR: 
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

tfr_group_data = {f'{a}': [] for a in rois}

power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids}

for subj_id in subj_ids: 
    if subj_id in ['MS024', 'MS034', 'MS038']:
        continue
    for event in analysis_evs:
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        
        save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/Day1'
        event_epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif', preload=True)
        ch_names = event_epochs_reref.ch_names
        #epochs_all_subjs_all_evs[subj_id][event]
            
        # Let's make sure we only do this for good channels
        good_chans = [x for x in event_epochs_reref.ch_names if x not in event_epochs_reref.info['bads']]
        picks = [x for x in good_chans]
        
        pow_struct = np.nan * np.ones([event_epochs_reref._data.shape[0], 
                               event_epochs_reref._data.shape[1], len(freqs), 
                               event_epochs_reref._data.shape[-1]])
    
        progress_bar = tqdm(np.arange(event_epochs_reref._data.shape[1]), ascii=True, desc='Computing TFR')

        for ch_ix in progress_bar: 
            ch_data = event_epochs_reref._data[:, ch_ix:ch_ix+1, :]
            bad_epochs  = np.where(event_epochs_reref.metadata[event_epochs_reref.ch_names[ch_ix]].notnull())[0]
            good_epochs = np.delete(np.arange(ch_data.shape[0]), bad_epochs)
            ch_data = np.delete(ch_data, bad_epochs, axis=0)
            ch_pow = mne.time_frequency.tfr_array_morlet(ch_data, sfreq=event_epochs_reref.info['sfreq'], 
                                                freqs=freqs, n_cycles=n_cycles, zero_mean=False, 
                                                use_fft=True, output='power', n_jobs=-1)

            pow_struct[good_epochs, ch_ix, :, :] = ch_pow[:, 0, :, :]
        
        temp_pow = mne.time_frequency.EpochsTFR(event_epochs_reref.info, pow_struct, 
                                                event_epochs_reref.times, freqs)
        
        temp_pow.crop(tmin=evs[event][0], tmax=evs[event][1])
        
        # Get electrode df 
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
        
        progress_bar = tqdm(elec_df.label, ascii=True, desc='Aggregating TFRs')

        for elec_ix, chan in enumerate(progress_bar):
            region = elec_df.salman_region.iloc[elec_ix]
            if region in rois:
                # average across channels, events, and times
                plot_data = np.nanmean(np.nanmean(np.nanmean(temp_pow.copy().pick_channels([chan]).data, axis=0), axis=0), axis=-1)
                tfr_group_data[region].append(plot_data)                    
                    

In [ ]:
freqs = np.logspace(*np.log10([2, 200]), num=30)
save_dir_plots = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBanditsEMU_Ephys'
os.makedirs(save_dir_plots, exist_ok=True)

for region in rois: 

    fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
    # Create a Pandas DataFrame
    plot_data = np.vstack(tfr_group_data[region])
    normalized_array = np.array([normalize_list(lst) for lst in plot_data])
    for ch in normalized_array:
        # plt.plot(freqs[:-1], np.log10(ch)[:-1], alpha=0.1, color='k')
        plot_power_spectra(freqs[:-1], ch[:-1], alpha =0.1,  
                       ax=ax, colors='black')   
    plot_power_spectra(freqs[:-1], np.nanmean(normalized_array[:, :-1], axis=0), 
                       ax=ax, colors='black')   
    # plt.plot(freqs[:-1], np.log10(np.nanmean(normalized_array[:, :-1], axis=0)), alpha=1, color='k')

    
    ax.set(title=f'{region}', xlabel='log frequency (hz)', 
           ylabel='log power (norm)', 
           xticks=[2, 10, 30, 50, 200], 
           xticklabels=[2, 10, 30, 50, 200])
    
    sns.despine()
    
    plt.savefig(f'{save_dir_plots}/recog_PSD_{region}.pdf', dpi=300)

        
        

 I have to plot the bayesian regression results in another noteboko that loads the proper environment for that type of analysis : BayesianRegression.ipynb